# DECOMP — Features deep-dive

What every data stream in an IBL session **is**, **how it should be processed**, and **what it looks like rendered**.

Focal session: `41431f53-69fd-4e3b-80ce-ea62e03bf9c7` (subject `CSH_ZAD_022`, Zador lab, 2 Neuropixels probes, ~88 min, 570 trials).

Sections:
1. Session card — metadata + recording timeline
2. **Probe anatomy** — where are the probes physically? Channel-by-region depth maps, CCF coordinates
3. **Clusters** — sorted units, amplitudes, depths, regions
4. **Spike rasters** — what 30 seconds of population activity looks like
5. **Trial structure** — per-trial Gantt of task events, block prior, performance
6. **Wheel** — rotary encoder → position / velocity / acceleration
7. **Camera + DLC pose** — markers in image coordinates, ROIs, optional video frame
8. **Pupil** — derived from 4 corneal markers, ellipse + diameter
9. **Licks** — derived from tongue markers, threshold-crossing events
10. **Motion energy** — whisker pad ROI scalar per frame
11. **Combined timeline** — all streams on one time axis

Each section starts with an **explainer paragraph**: what this feature is, what its time scale is, and how we process it for the GLM / SVCA / CCA pipeline.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, Ellipse
from matplotlib.collections import LineCollection

from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units

from decomp.viz.figures import _BASE_STYLE, _REGION_COLORS
mpl.rcParams.update(_BASE_STYLE)

EID = '41431f53-69fd-4e3b-80ce-ea62e03bf9c7'
one = ONE(base_url='https://openalyx.internationalbrainlab.org', password='international', silent=True)
bwm = bwm_query(one)
PIDS = bwm[bwm['eid'] == EID]['pid'].tolist()
print(f'Session {EID}, probes:', PIDS)

## 1. Session card

Quick metadata: subject, lab, date, task, probe count, trial count, recording duration. Comes from Alyx and from the trials table.

In [ ]:
session_meta = one.alyx.rest('sessions', 'read', id=EID)
info = {
    'subject': session_meta.get('subject'),
    'lab': session_meta.get('lab'),
    'date': session_meta.get('start_time', '')[:10],
    'task': session_meta.get('task_protocol'),
    'n_probes': len(PIDS),
}
for k, v in info.items():
    print(f'  {k:10s}: {v}')

sl = SessionLoader(one=one, eid=EID)
sl.load_trials()
trials = sl.trials
rec_duration = float(trials['intervals_1'].max() - trials['intervals_0'].min())
print(f'  trials   : {len(trials)} ({(trials["feedbackType"] == 1).mean():.0%} correct)')
print(f'  duration : {rec_duration / 60:.1f} min ({rec_duration:.0f} s)')
print(f'  median RT: {(trials["response_times"] - trials["stimOn_times"]).median():.2f} s')

## 2. Probe anatomy — where are the probes?

**What it is.** A Neuropixels probe is a thin silicon shank with 384 active recording channels arranged in a checkerboard pattern over its length. After insertion, IBL's pipeline runs spike-sorting (Kilosort / Pykilosort) to find single units, then performs *electrode localization* by aligning post-mortem histology to the Allen Common Coordinate Framework (CCF). The output: every channel and every cluster has 3D coordinates (`mlapdv` = medial-lateral, anterior-posterior, dorsal-ventral, in micrometers) and a Beryl-atlas region label.

**Why we care.** This is what tells us *which neurons are in V1 vs CB vs CA1.* The cluster's `atlas_id` mapped through `BrainRegions.id2acronym(..., mapping='Beryl')` gives the ROI we use for filtering.

**How to read the figure below.** Each probe is shown as a vertical strip; each row is a channel; row color = Beryl-atlas region the channel sits in. Top of the strip = brain surface (low DV). Bottom = deepest. The cluster scatter on the right shows individual sorted units' depths and amplitudes.

In [ ]:
# Pull channels + clusters per probe and build a region color map
from iblatlas.atlas import BrainRegions
br = BrainRegions()

probe_data = {}
for pid in PIDS:
    spikes, clusters = load_good_units(one, pid)
    # also grab raw channels to get full atlas info per channel (not just per cluster)
    try:
        channels = one.load_object(EID, 'channels', collection=f'alf/{spikes["probe_name"]}/pykilosort'
                                    if 'probe_name' in spikes else None)
    except Exception:
        channels = None
    clusters_df = pd.DataFrame(clusters)
    if 'Beryl' not in clusters_df.columns and 'atlas_id' in clusters_df.columns:
        clusters_df['Beryl'] = br.id2acronym(clusters_df['atlas_id'].to_numpy(), mapping='Beryl')
    probe_data[pid] = {'spikes': spikes, 'clusters': clusters_df}
    print(f'Probe {pid[:8]}: {len(clusters_df)} good units, regions: '
          f'{clusters_df["Beryl"].value_counts().head(8).to_dict()}')

In [ ]:
# Per-probe depth × region strip + cluster scatter
from itertools import cycle
from matplotlib.colors import to_hex

# Build region color palette dynamically (consistent across both probes).
# Convert every entry to a hex string so we never mix tuple/string when scatter takes c=array.
all_regions = sorted(set(
    r for d in probe_data.values() for r in d['clusters']['Beryl'].dropna().unique()
))
palette = [to_hex(c) for c in (plt.get_cmap('tab20').colors + plt.get_cmap('tab20b').colors)]
REGION_PAL = dict(zip(all_regions, cycle(palette)))
DEFAULT_HEX = '#888888'

fig, axes = plt.subplots(1, 2 * len(PIDS), figsize=(4.5 * len(PIDS), 7),
                          gridspec_kw={'width_ratios': [0.4, 1] * len(PIDS)})
if len(PIDS) == 1:
    axes = axes[:, None] if axes.ndim == 1 else axes
for j, (pid, d) in enumerate(probe_data.items()):
    clu = d['clusters']

    # Region strip: each cluster as a horizontal bar at its depth
    ax_strip = axes[2 * j]
    for _, row in clu.iterrows():
        c = REGION_PAL.get(row['Beryl'], DEFAULT_HEX)
        ax_strip.barh(row['depths'], 1.0, height=15, color=c, alpha=0.85, edgecolor='none')
    ax_strip.set_xlim(0, 1); ax_strip.set_xticks([])
    ax_strip.invert_yaxis()  # depth increases downward
    ax_strip.set_ylabel('Depth from probe tip (μm)')
    ax_strip.set_title(f'{pid[:8]}\n(n={len(clu)})', fontsize=12)
    ax_strip.spines['top'].set_visible(False); ax_strip.spines['right'].set_visible(False)

    # Cluster scatter: amplitude × depth, colored by region
    ax_sc = axes[2 * j + 1]
    for region, sub in clu.groupby('Beryl'):
        ax_sc.scatter(sub['amp_max'] * 1e6, sub['depths'], s=40,
                      color=REGION_PAL.get(region, DEFAULT_HEX), edgecolor='white',
                      linewidths=0.6, label=region, alpha=0.85)
    ax_sc.invert_yaxis()
    ax_sc.set_xlabel('Max waveform amplitude (μV)')
    ax_sc.set_xscale('log')
    ax_sc.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=False)
    ax_sc.spines['top'].set_visible(False); ax_sc.spines['right'].set_visible(False)
fig.suptitle('Probe anatomy: clusters × depth × region', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 3D scatter of cluster positions in CCF space (when histology coordinates are available)
fig = plt.figure(figsize=(11, 5.5))
for j, (pid, d) in enumerate(probe_data.items()):
    clu = d['clusters']
    if not all(c in clu.columns for c in ('x', 'y', 'z')):
        continue
    # IBL convention: x=ML, y=AP, z=DV in micrometers from bregma
    ax = fig.add_subplot(1, len(PIDS), j + 1, projection='3d')
    for region, sub in clu.groupby('Beryl'):
        ax.scatter(sub['x'] * 1e6, sub['y'] * 1e6, -sub['z'] * 1e6, s=30,
                   color=REGION_PAL.get(region, DEFAULT_HEX), edgecolor='white',
                   linewidths=0.4, label=region, alpha=0.85)
    ax.set_xlabel('ML (μm)'); ax.set_ylabel('AP (μm)'); ax.set_zlabel('DV (μm)')
    ax.set_title(f'Probe {pid[:8]}', fontsize=12)
    ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
fig.suptitle('Cluster positions in Allen CCF (ML × AP × DV)', y=1.02)
plt.tight_layout(); plt.show()

## 3. Cluster table — every sorted unit

**What it is.** Each row is one sorted neuron (cluster). Columns include `cluster_id` (integer in [0, n_clusters)), `depths` (μm from probe tip), `channels` (which channel had the largest spike on this unit), `amp_max/amp_min/amp_median` (waveform amplitudes), `atlas_id` and `Beryl` (anatomical location), and various QC metrics. We filter to `label == 1.0` (good QC).

**How to process for our pipeline.** Filter by Beryl region → assign to ROI (VISp / CB / MO / CA1) → grab spike times for that cluster from `spikes['times']` where `spikes['clusters'] == cluster_id`.

In [ ]:
# Concat both probes' clusters, show top 20 by max amplitude
combined = pd.concat(
    [d['clusters'].assign(pid=pid[:8]) for pid, d in probe_data.items()], ignore_index=True
)
rate = []
for pid, d in probe_data.items():
    clu_ids, counts = np.unique(d['spikes']['clusters'], return_counts=True)
    rec_t = d['spikes']['times'].max() - d['spikes']['times'].min()
    fr = pd.Series(counts / rec_t, index=clu_ids)
    sub = d['clusters'].copy(); sub['pid'] = pid[:8]
    sub['fr_hz'] = sub['cluster_id'].map(fr).fillna(0)
    rate.append(sub)
combined = pd.concat(rate, ignore_index=True)
show_cols = [c for c in ['pid', 'cluster_id', 'Beryl', 'depths', 'amp_max', 'fr_hz'] if c in combined.columns]
print(f'Total good-QC units across probes: {len(combined)}')
combined[show_cols].sort_values('amp_max', ascending=False).head(20)

## 4. Spike raster

**What it is.** Each row is one unit, each tick is one spike. Y-axis sorted by depth (or by region). Time on the X-axis. The classic neuroscience plot.

**Why we care.** Visual confirmation that the firing patterns are non-trivial — bursts, lulls, structured covariation across nearby neurons. If the raster looked like uniform Poisson noise, we'd halt and audit.

**Processing.** No binning, just plot spike times directly. We zoom to a 30 s window for legibility.

In [ ]:
# Pick a 30 s window in the middle of the session
WIN = (rec_duration / 2 - 15, rec_duration / 2 + 15)
fig, axes = plt.subplots(len(PIDS), 1, figsize=(11, 4 * len(PIDS)), sharex=True)
if len(PIDS) == 1:
    axes = [axes]
for ax, (pid, d) in zip(axes, probe_data.items()):
    spikes = d['spikes']; clu = d['clusters']
    cid_to_depth = dict(zip(clu['cluster_id'], clu['depths']))
    cid_to_region = dict(zip(clu['cluster_id'], clu['Beryl']))
    in_win = (spikes['times'] >= WIN[0]) & (spikes['times'] <= WIN[1])
    t = spikes['times'][in_win]
    c = spikes['clusters'][in_win]
    # y = cluster's depth, color = region (all colors are hex strings, so c=list-of-hex is fine)
    y = np.array([cid_to_depth.get(cid, np.nan) for cid in c])
    colors = [REGION_PAL.get(cid_to_region.get(cid, ''), DEFAULT_HEX) for cid in c]
    ax.scatter(t, y, s=1.5, c=colors, marker='|', linewidths=0.7)
    ax.invert_yaxis()
    ax.set_ylabel(f'Depth (μm)\nProbe {pid[:8]}', fontsize=11)
    ax.set_xlim(WIN)
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Spike raster, 30 s window — colors = Beryl regions', y=1.0)
plt.tight_layout(); plt.show()

## 5. Trial structure

**What it is.** The IBL contrast-discrimination task is a Pavlovian forced choice: a Gabor patch appears on the left or right at one of 5 contrasts (0, 0.0625, 0.125, 0.25, 1.0). The mouse rotates a wheel left or right to bring the stimulus to center; choice = direction. Reward (water) on correct, white-noise burst on error. Trials come in **blocks of 20–100 trials** with a fixed `probabilityLeft` (0.2, 0.5, or 0.8) — the block prior.

**The 20 columns of the trials table** time-stamp every relevant event:
- `goCueTrigger_times → goCue_times` — go-cue audio fires after a quiescence period
- `stimOnTrigger_times → stimOn_times` — visual stimulus appears (this is the canonical event-0 we align to)
- `firstMovement_times` — first detected wheel movement after stim
- `response_times` — wheel reaches threshold, choice committed
- `stimOff_times`, `feedback_times`, `feedbackType` — outcome
- `intervals_0`, `intervals_1` — trial start / end

**Processing.** Use `firstMovement_times` and `stimOn_times` as the GLM's movement and stimulus event regressors. Choice gets encoded as left vs right at `response_times`. Block prior enters as a step function of `probabilityLeft` keyed by trial start.

In [ ]:
# Per-trial Gantt of the first 20 trials
fig, ax = plt.subplots(figsize=(11, 7))
events = [
    ('Quiescence', 'intervals_0', 'goCue_times', '#cccccc'),
    ('Stim on',    'stimOn_times', 'firstMovement_times', '#3a7ca5'),
    ('Movement',   'firstMovement_times', 'response_times', '#2ca02c'),
    ('Feedback',   'response_times', 'feedback_times', '#c44536'),
    ('ITI',        'feedback_times', 'intervals_1', '#cccccc'),
]
for trial_idx in range(20):
    row = trials.iloc[trial_idx]
    t0 = row['intervals_0']
    for label, tcol_start, tcol_end, color in events:
        a, b = row.get(tcol_start, np.nan), row.get(tcol_end, np.nan)
        if pd.notnull(a) and pd.notnull(b):
            ax.barh(trial_idx, b - a, left=a - t0, color=color,
                    label=label if trial_idx == 0 else None, edgecolor='white', linewidth=0.5)
ax.set_xlim(0, 12)
ax.set_xlabel('Time within trial (s)')
ax.set_ylabel('Trial #')
ax.set_title('Per-trial event timeline (first 20 trials)')
ax.legend(loc='upper right', frameon=False)
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# Block prior over trials + smoothed correct rate
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(np.arange(len(trials)), trials['probabilityLeft'], color='#3a7ca5', lw=1.4)
axes[0].set_ylabel('p(Left) block prior'); axes[0].set_ylim(0, 1)
axes[0].set_title('Block prior over trials')
from scipy.ndimage import uniform_filter1d
correct = (trials['feedbackType'] == 1).astype(float).to_numpy()
smoothed = uniform_filter1d(correct, size=20, mode='nearest')
axes[1].plot(np.arange(len(trials)), smoothed, color='#2ca02c', lw=1.6)
axes[1].axhline(0.5, ls='--', color='#888', lw=1)
axes[1].set_ylabel('% correct (20-trial smooth)'); axes[1].set_xlabel('Trial #')
axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 6. Wheel

**What it is.** The mouse turns a 1" plastic wheel attached to a rotary encoder. The encoder produces sub-millisecond ticks every 1.27 mm of motion (encoder resolution). IBL ships:
- `wheel.position.npy`, `wheel.timestamps.npy` — raw encoder ticks (**irregular sample times**)
- `SessionLoader.load_wheel()` interpolates to a uniform **1 kHz grid** and applies an 8th-order Butterworth low-pass at 20 Hz to derive `velocity` (rad/s) and `acceleration` (rad/s²)

**Why we care.** Wheel velocity is the cleanest single scalar for locomotion. It's the most important regressor in the GLM's movement kernel and one of the two confounds in our pCCA.

**Processing.** Bin from 1 kHz down to our 20 ms grid by linear interpolation onto bin centers. The velocity sign tells us *direction* of running.

In [ ]:
sl.load_wheel()
wheel = sl.wheel
print(f'wheel: {len(wheel):,} samples at 1 kHz, columns: {list(wheel.columns)}')

# Position + velocity over a 60 s window
t0, t1 = WIN
m = (wheel['times'] >= t0) & (wheel['times'] <= t1)
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
axes[0].plot(wheel['times'][m], wheel['position'][m], color='#3a7ca5', lw=1.4)
axes[0].set_ylabel('Position (rad)'); axes[0].set_title(f'Wheel, 30 s window @ {t0:.0f}–{t1:.0f} s')
axes[1].plot(wheel['times'][m], wheel['velocity'][m], color='#2ca02c', lw=1.4)
axes[1].axhline(0, color='#888', lw=0.5)
axes[1].set_ylabel('Velocity (rad/s)')
axes[2].plot(wheel['times'][m], wheel['acceleration'][m], color='#c44536', lw=1.4)
axes[2].axhline(0, color='#888', lw=0.5)
axes[2].set_ylabel('Acceleration (rad/s²)'); axes[2].set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

# Velocity histogram (log y) + phase plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(wheel['velocity'].dropna(), bins=120, color='#2ca02c', edgecolor='white')
axes[0].set_yscale('log'); axes[0].set_xlabel('Velocity (rad/s)'); axes[0].set_ylabel('# samples (log)')
axes[0].set_title('Wheel velocity distribution (heavy-tailed)')
ds = max(1, len(wheel) // 50000)
axes[1].scatter(wheel['position'][::ds], wheel['velocity'][::ds], s=2, alpha=0.15, color='#3a7ca5')
axes[1].set_xlabel('Position (rad)'); axes[1].set_ylabel('Velocity (rad/s)')
axes[1].set_title('Position–velocity phase plot')
plt.tight_layout(); plt.show()

## 7. Camera + DLC pose markers

**What it is.** IBL records up to three head-fixed cameras: **leftCamera** (60 Hz, full-res, side view of left eye/face), **rightCamera** (150 Hz, half-res, side view of right eye/face), **bodyCamera** (30 Hz, top-down body view). On most older sessions only `leftCamera` is available. Each camera produces:
- `_iblrig_<cam>Camera.raw.mp4` — raw video file (~1.5 GB)
- `_ibl_<cam>Camera.times.npy` — wall-clock timestamp per frame, aligned to spikes
- `_ibl_<cam>Camera.dlc.pqt` and/or `.lightningPose.pqt` — frame-by-frame body-part positions in image coordinates
- `<cam>Camera.ROIMotionEnergy.npy` — scalar motion energy in a fixed ROI per frame

**Body parts tracked on leftCamera** (11 markers):
- 4 pupil markers: `pupil_top_r`, `pupil_right_r`, `pupil_bottom_r`, `pupil_left_r` (the `_r` suffix means right-eye-as-mirrored, since the left camera images the right eye after mirroring)
- `nose_tip`
- 2 paw markers: `paw_l`, `paw_r` (closest to the camera)
- 2 tongue markers: `tongue_end_l`, `tongue_end_r`
- 2 reward-tube markers: `tube_top`, `tube_bottom`

Each marker comes with `(x, y, likelihood)` triplets. We threshold likelihood ≥ 0.9 and fall through to NaN otherwise.

**Processing.** Pupil → derived ellipse from 4 pupil markers; lick events → tongue threshold-crossings; paws / nose → continuous regressors in the GLM.

In [ ]:
# Load pose for the left camera and show all markers in image coordinates
sl.pose = {}; sl.motion_energy = {}
for view in ('left',):
    try:
        sl.load_pose(views=[view], likelihood_thr=0.9)
        sl.load_motion_energy(views=[view])
    except Exception as e:
        print(f'{view} camera: {type(e).__name__}: {e}')
pose = sl.pose.get('leftCamera')
me = sl.motion_energy.get('leftCamera')
print(f'leftCamera pose: shape={pose.shape}, frame rate ~ 60 Hz')
print(f'leftCamera ME: shape={me.shape}')

In [15]:
# Marker scatter in image coordinates: median (x, y) per body part across the session.
# Falls back gracefully if the optional video frame load fails.
marker_groups = {
    'pupil': ['pupil_top_r', 'pupil_right_r', 'pupil_bottom_r', 'pupil_left_r'],
    'face':  ['nose_tip'],
    'paw':   ['paw_l', 'paw_r'],
    'tongue': ['tongue_end_l', 'tongue_end_r'],
    'tube':  ['tube_top', 'tube_bottom'],
}
marker_colors = {'pupil': '#7a5db8', 'face': '#3a7ca5', 'paw': '#2ca02c',
                  'tongue': '#c44536', 'tube': '#888888'}

### 7b. Real video frames at different behavioral states

The leftCamera mp4 (~3 GB) is already on disk for this session, so we can render actual frames. Below: 6 frames sampled at different behavioral moments (rest, running peak, stim onset, first lick, error feedback, late session), each with DLC markers overlaid in the same colors as the marker-scatter above.

This is the most direct way to verify that *the markers we feed into the GLM track real body parts on real frames*.

In [ ]:
import imageio.v3 as iio

cam_path = one.load_dataset(EID, '_iblrig_leftCamera.raw.mp4', download_only=True)
cam_times = pose['times'].to_numpy()  # one entry per frame, aligned to spike clock

def time_to_frame_idx(t_event):
    """Find the frame index whose camera-clock timestamp is closest to t_event."""
    return int(np.argmin(np.abs(cam_times - t_event)))

# Pick 6 informative time points -- using only data already loaded above
# 1. Resting: low wheel + low ME (quietest stretch in first half)
v = np.abs(np.interp(cam_times, wheel['times'], wheel['velocity']))
me_at_frames = np.interp(cam_times, me['times'], me['whiskerMotionEnergy'])
quiet_score = -(v + 5 * (me_at_frames - me_at_frames.min()))
i_rest = int(np.argmax(quiet_score[:len(cam_times) // 2]))

# 2. Running peak: highest wheel velocity
i_run = int(np.argmax(v))

# 3. First stim onset
i_stim = time_to_frame_idx(trials['stimOn_times'].dropna().iloc[0])

# 4. First response (mouse commits its choice — wheel-turn body movement)
i_resp = time_to_frame_idx(trials['response_times'].dropna().iloc[0])

# 5. First error feedback
err_times = trials.loc[trials['feedbackType'] == -1, 'feedback_times'].dropna()
i_err = time_to_frame_idx(err_times.iloc[0]) if len(err_times) else len(cam_times) // 2

# 6. Mid-session sample
i_mid = len(cam_times) // 2

picks = [
    (i_rest, 'rest'),
    (i_run,  'running peak'),
    (i_stim, 'first stim onset'),
    (i_resp, 'first response'),
    (i_err,  'first error fb'),
    (i_mid,  'mid-session'),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (frame_idx, label) in zip(axes.ravel(), picks):
    frame = iio.imread(cam_path, index=int(frame_idx))
    ax.imshow(frame, cmap='gray', aspect='equal')

    # Overlay every DLC marker at this frame, color-coded by group
    for group, parts in marker_groups.items():
        xs, ys = [], []
        for p in parts:
            x = pose[f'{p}_x'].iloc[frame_idx]
            y = pose[f'{p}_y'].iloc[frame_idx]
            if np.isfinite(x) and np.isfinite(y):
                xs.append(x); ys.append(y)
        if xs:
            ax.scatter(xs, ys, s=80, color=marker_colors[group], edgecolor='white',
                       linewidths=1.0, zorder=3)

    ax.set_title(f'frame {frame_idx}  ·  t={cam_times[frame_idx]:.1f}s  ·  {label}',
                 fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('leftCamera frames with DLC marker overlays', y=1.005, fontsize=16)
plt.tight_layout(); plt.show()

### 7c. Animation — 3 seconds of video with DLC tracking

Renders ~30 frames at 10 Hz (3 seconds of behavior at peak running speed) with markers tracking, written to `outputs/dlc_overlay.gif` and embedded inline. About 5–10 seconds to render.

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import Image as IPImage, display

# 3-second window centered on running peak, sampled every 6th frame (~10 Hz from 60 Hz source)
center = i_run
n_frames = 30
stride = 6
start_frame = max(0, center - (n_frames // 2) * stride)

gif_frames = []
for k in range(n_frames):
    fi = start_frame + k * stride
    if fi >= len(cam_times):
        break
    arr = iio.imread(cam_path, index=int(fi))
    if arr.ndim == 2:
        rgb = np.stack([arr] * 3, axis=-1)
    else:
        rgb = arr[..., :3]
    img = Image.fromarray(rgb.astype(np.uint8))
    draw = ImageDraw.Draw(img)
    for group, parts in marker_groups.items():
        col = tuple(int(255 * c) for c in mpl.colors.to_rgb(marker_colors[group]))
        for p in parts:
            x = pose[f'{p}_x'].iloc[fi]; y = pose[f'{p}_y'].iloc[fi]
            if np.isfinite(x) and np.isfinite(y):
                r = 6
                draw.ellipse([x - r, y - r, x + r, y + r], fill=col, outline=(255, 255, 255))
    gif_frames.append(img)

OUT_GIF = Path('../outputs/dlc_overlay.gif')
OUT_GIF.parent.mkdir(parents=True, exist_ok=True)
gif_frames[0].save(OUT_GIF, save_all=True, append_images=gif_frames[1:],
                    duration=100, loop=0, optimize=True)
print(f'wrote {OUT_GIF.resolve()} ({OUT_GIF.stat().st_size / 1e6:.1f} MB, {len(gif_frames)} frames)')
display(IPImage(str(OUT_GIF)))

## 8. Pupil — derived from 4 corneal markers

**What it is.** IBL doesn't ship `pupil.diameter` as a standalone dataset. Instead, the four pupil markers (top, right, bottom, left of the iris) are tracked by DLC; pupil diameter is derived as the **diagonal of the bounding box** formed by these four points (or via ellipse-fit when available). `brainbox.behavior.dlc.get_pupil_diameter` computes this; `get_smooth_pupil_diameter` adds a SNR-thresholded median filter.

**Why we care.** Pupil diameter is the standard scalar proxy for arousal in head-fixed mice. It's the second of our two confounds in the pCCA partialling step.

**Processing.** `SessionLoader.load_pupil()` returns `(times, pupilDiameter_raw, pupilDiameter_smooth)`. Interpolate onto our 20 ms bin grid for the GLM and pCCA.

In [ ]:
# Try the canonical pupil loader first; fall back to deriving from markers if it fails
pupil = None
try:
    sl.load_pupil(snr_thresh=5.0)
    pupil = sl.pupil
except Exception:
    pass
if pupil is None or len(pupil) == 0:
    from brainbox.behavior.dlc import get_pupil_diameter, get_smooth_pupil_diameter
    d = get_pupil_diameter(pose)
    ds = get_smooth_pupil_diameter(d, 'left')
    pupil = pd.DataFrame({'times': pose['times'].to_numpy(),
                          'pupilDiameter_raw': d, 'pupilDiameter_smooth': ds})
print(f'pupil: shape={pupil.shape}, columns: {list(pupil.columns)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# 4-marker pupil ellipse (median)
ax = axes[0]
for p in ['pupil_top_r', 'pupil_right_r', 'pupil_bottom_r', 'pupil_left_r']:
    ax.scatter(pose[f'{p}_x'].median(), pose[f'{p}_y'].median(), s=160, color='#7a5db8',
               edgecolor='white', linewidth=1.4, zorder=3)
    ax.annotate(p.replace('pupil_', '').replace('_r', ''),
                (pose[f'{p}_x'].median(), pose[f'{p}_y'].median()),
                xytext=(8, 0), textcoords='offset points', fontsize=9, color='#7a5db8')
x_med = np.array([pose[f'{p}_x'].median() for p in ['pupil_top_r', 'pupil_right_r',
                                                     'pupil_bottom_r', 'pupil_left_r']])
y_med = np.array([pose[f'{p}_y'].median() for p in ['pupil_top_r', 'pupil_right_r',
                                                     'pupil_bottom_r', 'pupil_left_r']])
cx, cy = x_med.mean(), y_med.mean()
rx = (x_med.max() - x_med.min()) / 2
ry = (y_med.max() - y_med.min()) / 2
ax.add_patch(Ellipse((cx, cy), 2 * rx, 2 * ry, fc='none', ec='#7a5db8', lw=2))
ax.set_aspect('equal'); ax.invert_yaxis()
ax.set_title('Pupil ellipse from 4 corneal markers')
ax.set_xlabel('image x'); ax.set_ylabel('image y')

# Pupil diameter time series
ax = axes[1]
ax.plot(pupil['times'], pupil['pupilDiameter_raw'], color='#7a5db8', alpha=0.4, lw=0.6,
        label='raw')
if 'pupilDiameter_smooth' in pupil.columns:
    ax.plot(pupil['times'], pupil['pupilDiameter_smooth'], color='#7a5db8', lw=1.6,
            label='smoothed')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Pupil diameter (px)')
ax.set_title('Pupil diameter over the session (slow drift = arousal state)')
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 9. Licks — derived from tongue markers

**What it is.** When the mouse licks the reward tube, the tongue extends past the lips; the DLC tracker picks it up briefly. `brainbox.behavior.dlc.get_licks(dlc, dlc_t)` detects threshold-crossings on the `tongue_end_l_*` and `tongue_end_r_*` traces and returns lick event timestamps.

**Why we care.** Licks are the principal consumption-related body movement and contribute meaningfully to the GLM's movement kernel. Pure motion energy doesn't disambiguate licks from whisking or paw movement.

**Processing.** Convert to a per-bin lick rate (events / s) at the 20 ms grid; this is one column of the design matrix. We could also event-align to look at neural response around licks, but in our pipeline we use the rate.

In [ ]:
from brainbox.behavior.dlc import get_licks
lick_times = get_licks(pose, pose['times'].to_numpy())
print(f'Detected {len(lick_times)} lick events ({len(lick_times) / (rec_duration / 60):.1f} per min)')

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
# Tongue y-coordinate over a 30 s window with lick-event markers
m = (pose['times'] >= WIN[0]) & (pose['times'] <= WIN[1])
tongue_y = pose.loc[m, 'tongue_end_l_y'].to_numpy()
axes[0].plot(pose['times'][m], tongue_y, color='#888', lw=1.0)
for t in lick_times[(lick_times >= WIN[0]) & (lick_times <= WIN[1])]:
    axes[0].axvline(t, color='#c44536', lw=1.0, alpha=0.7)
axes[0].set_ylabel('tongue_end_l_y (px)\n(lower = extended)')
axes[0].set_title('Tongue y-position with detected lick events (red lines)')
axes[0].invert_yaxis()
# Lick rate over the full session, smoothed
from scipy.ndimage import gaussian_filter1d
bin_edges = np.arange(0, rec_duration + 1, 1.0)
rates, _ = np.histogram(lick_times, bins=bin_edges)
axes[1].plot(bin_edges[:-1], gaussian_filter1d(rates, sigma=10), color='#c44536', lw=1.4)
axes[1].set_xlim(WIN)
axes[1].set_ylabel('Lick rate (Hz, smoothed)'); axes[1].set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

## 10. Motion energy — whisker-pad ROI

**What it is.** A single scalar per video frame: the mean absolute frame-difference inside a fixed bounding box around the whisker pad (left/right cameras) or the body (body camera). High value = mouse is moving its whiskers / face; low value = still.

**What it's NOT.** This is *one number per frame*. It is **not** the multi-component facemap SVDs (Stringer 2019, Syeda 2024) that track 16 different motion modes. Those would require running facemap on the raw video, which is out of scope for our pipeline.

**Why we care.** The whisker ME picks up uninstructed face movements (whisking, blinking, twitches). It's a small but real movement-related regressor in the GLM, particularly for V1 where face movements correlate with arousal-driven gain modulation.

**Processing.** Linear-interpolate from the camera frame rate (60 Hz) to our 20 ms grid; pass straight into the design matrix as a continuous covariate.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=False)
# ME over 30 s window
m = (me['times'] >= WIN[0]) & (me['times'] <= WIN[1])
axes[0].plot(me['times'][m], me['whiskerMotionEnergy'][m], color='#ffcc00', lw=1.4)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('whisker ME (a.u.)')
axes[0].set_title(f'Whisker-pad motion energy, 30 s window ({len(me):,} frames at 60 Hz)')
# ME vs |wheel velocity| — are they redundant?
from scipy.interpolate import interp1d
f = interp1d(wheel['times'], wheel['velocity'], bounds_error=False, fill_value=0)
wv_at_frames = f(me['times'].to_numpy())
abs_v = np.abs(wv_at_frames)
axes[1].scatter(abs_v[::10], me['whiskerMotionEnergy'][::10], s=3, alpha=0.2, color='#ffcc00')
axes[1].set_xlabel('|wheel velocity| (rad/s)'); axes[1].set_ylabel('whisker ME (a.u.)')
axes[1].set_title('Whisker ME vs running speed (correlated but not redundant)')
plt.tight_layout(); plt.show()

r = np.corrcoef(abs_v[np.isfinite(abs_v) & np.isfinite(me['whiskerMotionEnergy'])],
                 me['whiskerMotionEnergy'][np.isfinite(abs_v) & np.isfinite(me['whiskerMotionEnergy'])])[0, 1]
print(f'Pearson r(|wheel velocity|, whisker ME) = {r:.3f}')

## 11. Combined timeline — everything on one clock

**Why this view matters.** Our GLM / SVCA / CCA pipeline aligns *every* stream onto the same 20 ms bin grid. Eyeballing all of them on one time axis tells you whether the alignment makes physical sense: do spike bursts coincide with high motion energy? does pupil dilate when the mouse runs? do licks happen around feedback events?

If the timeline looks coherent (running periods have whisking, dilated pupils, neural bursts), our pipeline is operating on real, well-aligned signal.

In [ ]:
# Six-row stack: two probe spike counts + wheel + ME + pupil + lick events, all on the same 30 s window
from scipy.ndimage import gaussian_filter1d
from numpy import histogram

fig, axes = plt.subplots(6, 1, figsize=(11, 9), sharex=True,
                          gridspec_kw={'height_ratios': [1.5, 1.5, 1, 1, 1, 0.4]})
BIN = 0.05  # 50 ms population-rate bins
edges = np.arange(WIN[0], WIN[1] + BIN, BIN)

# (0, 1) per-probe population firing rate
for ax, (pid, d) in zip(axes[:2], probe_data.items()):
    counts, _ = np.histogram(d['spikes']['times'], bins=edges)
    rate = gaussian_filter1d(counts / BIN / len(d['clusters']), sigma=2)
    ax.plot(edges[:-1], rate, color='#3a7ca5', lw=1.4)
    ax.set_ylabel(f'{pid[:8]}\nrate (Hz/cell)')

# (2) wheel velocity
m = (wheel['times'] >= WIN[0]) & (wheel['times'] <= WIN[1])
axes[2].plot(wheel['times'][m], wheel['velocity'][m], color='#2ca02c', lw=1.2)
axes[2].axhline(0, color='#888', lw=0.5)
axes[2].set_ylabel('wheel\nvel (rad/s)')

# (3) motion energy
m = (me['times'] >= WIN[0]) & (me['times'] <= WIN[1])
axes[3].plot(me['times'][m], me['whiskerMotionEnergy'][m], color='#ffcc00', lw=1.2)
axes[3].set_ylabel('whisker\nME')

# (4) pupil
m = (pupil['times'] >= WIN[0]) & (pupil['times'] <= WIN[1])
col = 'pupilDiameter_smooth' if 'pupilDiameter_smooth' in pupil.columns else 'pupilDiameter_raw'
axes[4].plot(pupil['times'][m], pupil[col][m], color='#7a5db8', lw=1.2)
axes[4].set_ylabel('pupil\n(px)')

# (5) lick events as vertical ticks
axes[5].vlines(lick_times[(lick_times >= WIN[0]) & (lick_times <= WIN[1])], 0, 1,
               color='#c44536', lw=1.2)
axes[5].set_ylim(0, 1); axes[5].set_yticks([])
axes[5].set_ylabel('licks'); axes[5].set_xlabel('Time (s)')

# Overlay trial events as faint vertical lines on all axes
for ax in axes:
    for t in trials.loc[(trials['stimOn_times'] >= WIN[0]) &
                         (trials['stimOn_times'] <= WIN[1]), 'stimOn_times']:
        ax.axvline(t, color='#aaa', lw=0.5, alpha=0.6, zorder=0)

fig.suptitle('All streams on one 30 s window — gray verticals = stim onsets', y=1.005)
plt.tight_layout(); plt.show()

## How features compose into the pipeline

Given everything above, the way each stream feeds the GLM / SVCA / CCA / pCCA:

| Stream | Time scale | Used by GLM as | Used by SVCA / CCA |
|---|---|---|---|
| Spike times | sub-ms | Y (binned to 20 ms counts per neuron) | Y (same) |
| Trials: stim | event-aligned | event regressor + raised-cosine bases (5 kernels, 0.4 s) | — |
| Trials: first movement | event | event regressor + raised-cosine (5 kernels, ±0.2 s) | — |
| Trials: feedback / choice | event | event regressors | — |
| `probabilityLeft` | step / trial | step function (1 column) | — |
| Wheel velocity | 1 kHz → 20 ms | continuous + lagged bases | confound Z (in pCCA) |
| Wheel acceleration | 1 kHz → 20 ms | continuous + lagged bases | — |
| Whisker motion energy | 60 Hz → 20 ms | continuous + lagged bases | — |
| Pupil diameter | 60 Hz → 20 ms | continuous + lagged bases (3 kernels) | confound Z (in pCCA) |
| Lick rate | derived events → 20 ms count | continuous (rate per bin) | — |
| DLC paw / nose | 60 Hz → 20 ms | (currently unused; reserved) | — |
| Atlas / cluster region | static | filter Y to ROI {VISp, CB, MO, CA1} | filter Y to ROI |

**One thing that's not in the design matrix yet:** DLC paw and nose positions, and (when available) bodyCamera ME. Both are in `pose` and `motion_energy` and would be straightforward to add as continuous regressors — this is what "richer Z" buys us in the next-steps plan.

**One thing that's deliberately omitted:** raw video. We'd need facemap-style multi-component motion SVDs to use raw video meaningfully, and that's out of scope for the MVP.